# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [ ]:
pip install shap

In [ ]:
# Load the libraries as required.
from sklearn.preprocessing import StandardScaler, PowerTransformer, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error, make_scorer
import pandas as pd
import numpy as np
import joblib
import shap
import matplotlib.pyplot as plt

In [2]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [3]:
# Target variable
target_col = 'area'

y = fires_dt[target_col]


In [4]:
# Features (all columns except target)
X = fires_dt.drop(columns=[target_col])

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [5]:
# Identify numeric and categorical features
numeric_features = ['coord_x', 'coord_y', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']
categorical_features = ['month', 'day']

# Create the preprocessing pipeline
preproc1 = ColumnTransformer(transformers=[
    ('num', Pipeline(steps=[
        ('scaler', StandardScaler())
    ]), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])


### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [6]:
from sklearn.preprocessing import PowerTransformer

# Select variables for transformation
transform_vars = ['isi', 'dc']
non_transform_vars = list(set(numeric_features) - set(transform_vars))

# Create numeric pipeline
numeric_pipeline = ColumnTransformer(transformers=[
    # Apply power transform to selected variables
    ('power', Pipeline([
        ('scaler', StandardScaler()),
        ('power', PowerTransformer())
    ]), transform_vars),
    
    # Apply standard scaling to the rest
    ('scale', Pipeline([
        ('scaler', StandardScaler())
    ]), non_transform_vars)
])

# Full preproc2 with categorical encoding
preproc2 = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])


## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [12]:
# Pipeline A = preproc1 + baseline

from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, cross_val_score
from sklearn.metrics import mean_squared_error, make_scorer
import numpy as np

# Baseline model pipeline with preproc1 and Ridge regression
pipeline_a = Pipeline(steps=[
    ('preprocessing', preproc1),
    ('regressor', Ridge())
])



In [13]:
# Pipeline B = preproc2 + baseline


# Define Pipeline B with preproc2 and Ridge regressor
pipeline_b = Pipeline(steps=[
    ('preprocessing', preproc2),
    ('regressor', Ridge())
])





In [14]:
# Pipeline C = preproc1 + advanced model

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline

# Define Pipeline C
pipeline_c = Pipeline(steps=[
    ('preprocessing', preproc1),
    ('regressor', GradientBoostingRegressor(random_state=42))
])



In [15]:
# Pipeline D = preproc2 + advanced model

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.pipeline import Pipeline

# Define Pipeline D
pipeline_d = Pipeline(steps=[
    ('preprocessing', preproc2),
    ('regressor', GradientBoostingRegressor(random_state=42))
])


# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [16]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error, make_scorer
import numpy as np

# RMSE scorer
rmse_scorer = make_scorer(mean_squared_error, greater_is_better=False)


In [17]:
param_grid_a = {
    'regressor__alpha': [0.01, 0.1, 1.0, 10.0]
}

grid_a = GridSearchCV(pipeline_a, param_grid=param_grid_a, cv=10, scoring=rmse_scorer, n_jobs=-1)
grid_a.fit(X, y)
print(f"[Pipeline A] Best RMSE: {np.sqrt(-grid_a.best_score_):.4f}, Best Params: {grid_a.best_params_}")


[Pipeline A] Best RMSE: 65.1554, Best Params: {'regressor__alpha': 10.0}


In [18]:
param_grid_b = {
    'regressor__alpha': [0.01, 0.1, 1.0, 10.0]
}

grid_b = GridSearchCV(pipeline_b, param_grid=param_grid_b, cv=10, scoring=rmse_scorer, n_jobs=-1)
grid_b.fit(X, y)
print(f"[Pipeline B] Best RMSE: {np.sqrt(-grid_b.best_score_):.4f}, Best Params: {grid_b.best_params_}")


[Pipeline B] Best RMSE: 65.1809, Best Params: {'regressor__alpha': 10.0}


In [19]:
param_grid_c = {
    'regressor__n_estimators': [50, 100],
    'regressor__learning_rate': [0.05, 0.1]
}

grid_c = GridSearchCV(pipeline_c, param_grid=param_grid_c, cv=10, scoring=rmse_scorer, n_jobs=-1)
grid_c.fit(X, y)
print(f"[Pipeline C] Best RMSE: {np.sqrt(-grid_c.best_score_):.4f}, Best Params: {grid_c.best_params_}")


[Pipeline C] Best RMSE: 69.8582, Best Params: {'regressor__learning_rate': 0.1, 'regressor__n_estimators': 50}


In [20]:
param_grid_d = {
    'regressor__n_estimators': [50, 100],
    'regressor__learning_rate': [0.05, 0.1]
}

grid_d = GridSearchCV(pipeline_d, param_grid=param_grid_d, cv=10, scoring=rmse_scorer, n_jobs=-1)
grid_d.fit(X, y)
print(f"[Pipeline D] Best RMSE: {np.sqrt(-grid_d.best_score_):.4f}, Best Params: {grid_d.best_params_}")


[Pipeline D] Best RMSE: 69.9181, Best Params: {'regressor__learning_rate': 0.05, 'regressor__n_estimators': 50}


# Evaluate

+ Which model has the best performance?

The best model is Pipeline A based on the lowest RMSE. 

# Export

+ Save the best performing model to a pickle file.

In [21]:
import joblib

# Save Pipeline A's best model
joblib.dump(grid_a.best_estimator_, 'best_model_pipeline_a.pkl')

print("Best model (Pipeline A) saved as 'best_model_pipeline_a.pkl'")


Best model (Pipeline A) saved as 'best_model_pipeline_a.pkl'


# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [ ]:
import shap
# Extract preprocessing and model
preprocessor = grid_a.best_estimator_.named_steps['preprocessing']
model = grid_a.best_estimator_.named_steps['regressor']

# Transform X using preprocessing only
X_transformed = preprocessor.transform(X)

# Get feature names after one-hot encoding
feature_names = preprocessor.get_feature_names_out()
X_df = pd.DataFrame(X_transformed, columns=feature_names)


explainer = shap.Explainer(model.predict, X_df)
shap_values = explainer(X_df)


shap.plots.waterfall(shap_values[10], max_display=10)


# Summary bar plot (importance of features)
shap.plots.bar(shap_values, max_display=10)

# Or a dot plot with feature impact and distribution
shap.plots.beeswarm(shap_values, max_display=10)

# Drop low-importance features
X_reduced = X.drop(columns=['wind', 'rain'])  # example







# In general, across the complete training set, which features are the most and least important.
num__dc	+6.61	Drought Code (DC): Strongest impact on prediction, indicating fire potential.

# If you were to remove features from the model, which ones would you remove? Why?
1 num__rh – Relative humidity: Very low SHAP values overall. Narrow range of SHAP contributions across observations. Does not add much predictive signal.

2 cat__day_fri (and potentially other cat__day_* features): Categorical day-of-week appears weakly predictive. SHAP summary shows small, noisy contributions. Fire risk is unlikely to vary meaningfully by day of the week.

# How would you test that these features are actually enhancing model performance?
1 Create a Reduced Feature Set
2 Re-run Preprocessing + Modeling Pipeline
3 Run GridSearchCV Again
4 Compare Metrics

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.